# 🔬 Notebook 3: Flash Sale — Deep Dive (bad → best)


## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window:
`Cmd+Shift+P` → **Reload Window**.

This lab uses **only pure Python** (plus `pydantic`) — no Docker, no external services.
We simulate Redis, queues, and threads in-process so every cell runs anywhere.


## What we'll build, step by step

We'll implement the hot "reserve" path **four times**, getting better each time:

| Version | What it shows | Problem it has |
|---|---|---|
| 🟥 V1. **Naive** | `if stock > 0: stock -= 1` | **Oversells** under concurrency |
| 🟧 V2. **Locked** | Python `Lock` around check + decrement | Correct, but serial ≈ slow |
| 🟨 V3. **Atomic CAS (Lua-style)** | Single atomic op, like Redis Lua | Correct **and** fast |
| 🟩 V4. **Full path** | Atomic + per-user cap + TTL + admission queue | Production-shaped |

Every version runs on plain threads — no external services.


## 🟥 V1 — the naive overselling bug

This is the first implementation every junior engineer writes. It looks right. It isn't.


In [1]:
import threading, time, random

class NaiveStock:
    def __init__(self, initial: int): self.n = initial
    def try_reserve(self) -> bool:
        # 🟥 TOCTOU: check and modify are two separate ops — another thread sneaks in between
        if self.n > 0:
            time.sleep(0.00001)  # tiny pause amplifies the race (it exists without it too)
            self.n -= 1
            return True
        return False


def run_buyers(stock, n_threads=200, per_thread=20):
    reserved = [0]
    def worker():
        for _ in range(per_thread):
            if stock.try_reserve(): reserved[0] += 1
    ts = [threading.Thread(target=worker) for _ in range(n_threads)]
    for t in ts: t.start()
    for t in ts: t.join()
    return reserved[0]


INITIAL = 1000
stock = NaiveStock(INITIAL)
sold = run_buyers(stock)
print(f"Initial stock : {INITIAL}")
print(f"Reserved      : {sold}")
print(f"Stock counter : {stock.n}")
print(f"Oversold by   : {max(0, sold - INITIAL)} units "
      f"(or undersold by {max(0, INITIAL - sold)})")
print("→ In a real flash sale, oversold = angry customers + refunds + PR problem.")


Initial stock : 1000
Reserved      : 1029
Stock counter : -29
Oversold by   : 29 units (or undersold by 0)
→ In a real flash sale, oversold = angry customers + refunds + PR problem.


Even if the counter looks "about right" in a quick run, **it is not correct**. The bug is
real — it depends on thread scheduling. In production with thousands of concurrent workers,
oversold rows will happen. Never ship V1.


## 🟧 V2 — lock the check-and-decrement

The classic fix: put a lock around the whole operation so only one thread is inside at a time.
Correct, but serializes all buyers on one mutex.


In [2]:
class LockedStock:
    def __init__(self, initial: int):
        self.n = initial
        self.lock = threading.Lock()
    def try_reserve(self) -> bool:
        with self.lock:  # 🟧 whole region is atomic
            if self.n > 0:
                self.n -= 1
                return True
            return False


stock = LockedStock(INITIAL)
t0 = time.time()
sold = run_buyers(stock, n_threads=200, per_thread=20)
dt = time.time() - t0
print(f"Reserved      : {sold} / {INITIAL}")
print(f"Stock counter : {stock.n}")
print(f"Elapsed       : {dt*1000:.1f} ms")
assert sold == INITIAL and stock.n == 0, "must never oversell"
print("✓ no overselling — but all 200 threads queue on one lock")


Reserved      : 1000 / 1000
Stock counter : 0
Elapsed       : 10.2 ms
✓ no overselling — but all 200 threads queue on one lock


## 🟨 V3 — atomic CAS (what Redis Lua gives you)

Real Redis runs commands one at a time on a single thread. So a Lua script like:

```lua
-- KEYS[1] = "stock:item-42"
local remaining = tonumber(redis.call('GET', KEYS[1]))
if remaining and remaining > 0 then
  redis.call('DECR', KEYS[1])
  return 1
else
  return 0
end
```

…executes **atomically** across the entire cluster shard. No lock, no network round-trips
for GET/DECR separately. In Python we can simulate the same semantics with
`itertools.count` or `threading.Lock`. The important *concept* is:

> **One atomic primitive** — not "lock + two ops".

When you scale out beyond one Redis, you **shard the key** (`stock:item-42:shard{0..15}`),
and each shard runs its own atomic primitive. Global throughput = shards × per-shard RPS.


In [3]:
from itertools import count

class ShardedAtomicStock:
    """Simulates `stock:item-42:shard{0..N-1}` across N Redis shards."""
    def __init__(self, initial: int, shards: int = 8):
        base, extra = divmod(initial, shards)
        # Each shard gets (initial / shards); first `extra` shards get one more
        self.shards = [
            _AtomicCounter(base + (1 if i < extra else 0))
            for i in range(shards)
        ]
        self._rr = count()  # round-robin picker
    def try_reserve(self, user_id: int) -> bool:
        # Pick shard. Real systems use hash(user_id) % N for stickiness.
        idx = user_id % len(self.shards)
        # Fall-through: if our shard is empty, try the rest once (rare, but fair)
        for offset in range(len(self.shards)):
            s = self.shards[(idx + offset) % len(self.shards)]
            if s.decrement_if_positive():
                return True
        return False
    @property
    def total(self) -> int:
        return sum(s.value for s in self.shards)


class _AtomicCounter:
    """Represents one Redis shard running an atomic Lua script."""
    def __init__(self, initial: int):
        self._value = initial
        self._lock = threading.Lock()  # stands in for Redis's single-threaded executor
    @property
    def value(self) -> int: return self._value
    def decrement_if_positive(self) -> bool:
        with self._lock:
            if self._value > 0:
                self._value -= 1
                return True
            return False


stock = ShardedAtomicStock(INITIAL, shards=8)

reserved = [0]
def worker(uid_base: int):
    for i in range(20):
        if stock.try_reserve(user_id=uid_base + i):
            reserved[0] += 1

ts = [threading.Thread(target=worker, args=(k*1000,)) for k in range(200)]
t0 = time.time()
for t in ts: t.start()
for t in ts: t.join()
dt = time.time() - t0

print(f"Reserved      : {reserved[0]} / {INITIAL}")
print(f"Stock per shard: {[s.value for s in stock.shards]}")
print(f"Total left    : {stock.total}")
print(f"Elapsed       : {dt*1000:.1f} ms  (vs locked version above)")
assert reserved[0] == INITIAL and stock.total == 0
print("✓ no overselling, and 8 shards mean ~8x the parallelism")


Reserved      : 1000 / 1000
Stock per shard: [0, 0, 0, 0, 0, 0, 0, 0]
Total left    : 0
Elapsed       : 11.1 ms  (vs locked version above)
✓ no overselling, and 8 shards mean ~8x the parallelism


## 🛡️ The token-bucket rate limiter

Before stock, you want to drop obvious abusers. Token bucket is the industry default:

- Each user has a bucket with `capacity` tokens.
- Tokens refill at `rate` per second.
- Each request costs 1 token; if none left → **429 Too Many Requests**.

This absorbs small bursts (you probably click the button twice) while capping long-term rate.


In [4]:
class TokenBucket:
    def __init__(self, capacity: int, refill_per_s: float):
        self.capacity = capacity
        self.tokens = float(capacity)
        self.rate = refill_per_s
        self.last = time.time()
        self.lock = threading.Lock()

    def allow(self, cost: int = 1) -> bool:
        with self.lock:
            now = time.time()
            # Refill based on elapsed time (no background thread needed)
            self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
            self.last = now
            if self.tokens >= cost:
                self.tokens -= cost
                return True
            return False


# 2 requests per second sustained, with a burst of 3
tb = TokenBucket(capacity=3, refill_per_s=2)

# Scenario A — hammer it with no delay: expect first 3 to pass, rest rejected
burst = ["✓" if tb.allow() else "✗" for _ in range(10)]
print("burst (no delay)        :", " ".join(burst))

# Scenario B — back off and try again one at a time
time.sleep(0.6)
print("after 600ms pause       :", "✓" if tb.allow() else "✗", "(refill gave us ~1 token back)")
print("→ first 3 pass (burst capacity), extras rejected, tokens refill over time")


burst (no delay)        : ✓ ✓ ✓ ✗ ✗ ✗ ✗ ✗ ✗ ✗


after 600ms pause       : ✓ (refill gave us ~1 token back)
→ first 3 pass (burst capacity), extras rejected, tokens refill over time


## 📥 Admission queue with backpressure

Even with atomic stock and a rate limiter, 1M users hitting Redis directly is wasteful.
The answer: a **bounded queue**.

- Producers (API servers) enqueue reserve-requests.
- Consumers (workers) drain at a safe rate into Redis.
- Queue full → **immediate 503** or sold-out page, no origin work.

In real deployments this is Kafka, a Redis Stream, or AWS SQS. The behavior we care about
is the same: **bounded capacity + non-blocking enqueue**.


In [5]:
import queue


class BoundedAdmission:
    """Drop-on-full bounded queue. Mirrors Kafka/Redis-Stream semantics for this demo."""
    def __init__(self, maxsize: int):
        self.q: queue.Queue = queue.Queue(maxsize=maxsize)
        self.dropped = 0
    def offer(self, item) -> bool:
        try:
            self.q.put_nowait(item)  # non-blocking
            return True
        except queue.Full:
            self.dropped += 1        # caller gets 503 fast
            return False


def simulate(n_users: int, stock_size: int, queue_capacity: int):
    adm = BoundedAdmission(maxsize=queue_capacity)
    stock = ShardedAtomicStock(stock_size, shards=4)
    reserved, admitted = 0, 0

    # Phase 1 — everyone tries to get in
    for uid in range(n_users):
        if adm.offer(uid): admitted += 1

    # Phase 2 — workers drain the queue into stock
    while not adm.q.empty():
        uid = adm.q.get_nowait()
        if stock.try_reserve(user_id=uid): reserved += 1

    return {
        "users": n_users,
        "stock_size": stock_size,
        "queue_capacity": queue_capacity,
        "admitted": admitted,
        "dropped_fast": adm.dropped,
        "reserved": reserved,
        "sold_out_after_admission": admitted - reserved,
    }


# Key insight: sizing the queue ≈ 1.5× stock keeps most losers out with a fast "no"
for qcap in (500, 1500, 5000):
    result = simulate(n_users=50_000, stock_size=1_000, queue_capacity=qcap)
    print(result)


{'users': 50000, 'stock_size': 1000, 'queue_capacity': 500, 'admitted': 500, 'dropped_fast': 49500, 'reserved': 500, 'sold_out_after_admission': 0}
{'users': 50000, 'stock_size': 1000, 'queue_capacity': 1500, 'admitted': 1500, 'dropped_fast': 48500, 'reserved': 1000, 'sold_out_after_admission': 500}
{'users': 50000, 'stock_size': 1000, 'queue_capacity': 5000, 'admitted': 5000, 'dropped_fast': 45000, 'reserved': 1000, 'sold_out_after_admission': 4000}


### What the three rows above teach us

- `queue_capacity=500` — **too small**: fewer losers wait around, but the queue fills so fast
  that it acts as a random sampler. Some users who arrived in time still get dropped.
- `queue_capacity=1500` — **sweet spot**: ~1.5× stock. Anyone admitted has a realistic chance;
  everyone else gets their "sold out" page in milliseconds.
- `queue_capacity=5000` — **too big**: 5,000 users are told "keep waiting" only to discover 5
  minutes later that 4,000 of them lost. Terrible UX; wasted backend cycles.


## ⏳ Reservation TTL — how stock gets returned

Stock is reserved *before* payment. If the user never pays (tab closed, card declined, bot),
the stock must come back. The standard trick: **TTL + reaper**.

1. On `/reserve`: write `reservation:{id} = {user, item, qty, status=pending}` with `EX=600` (10 min).
2. On `/pay` success: update status to `paid`, drop the TTL (`PERSIST`).
3. A reaper job periodically scans expired reservations and **returns stock** for any still
   in `pending`.

The interesting correctness question: what if the reaper and the payment race?

- `paid` wins: pay first updates status, reaper sees `paid`, does nothing.
- `expired` wins: reaper decrements the `paid` counter back; payment call should then fail
  with **410 Gone** — the frontend shows "your seat was released, please try again."

Implementing this cleanly is a **compare-and-set** on the reservation's status field.


In [6]:
class ReservationStore:
    def __init__(self, stock: "ShardedAtomicStock"):
        self.stock = stock
        self.reservations: dict[str, dict] = {}
        self.lock = threading.Lock()

    def reserve(self, rid: str, user_id: int, ttl_s: float) -> bool:
        if not self.stock.try_reserve(user_id=user_id):
            return False
        expiry = time.time() + ttl_s
        with self.lock:
            self.reservations[rid] = {"status": "pending", "expires": expiry,
                                       "user": user_id}
        return True

    def pay(self, rid: str) -> str:
        with self.lock:
            r = self.reservations.get(rid)
            if not r: return "not_found"
            if r["status"] == "expired": return "expired"
            if time.time() > r["expires"]:
                r["status"] = "expired"; return "expired"
            r["status"] = "paid"
            return "paid"

    def reap(self) -> int:
        returned = 0
        now = time.time()
        with self.lock:
            for rid, r in list(self.reservations.items()):
                if r["status"] == "pending" and now > r["expires"]:
                    r["status"] = "expired"
                    # Return stock to its shard
                    self.stock.shards[r["user"] % len(self.stock.shards)]._value += 1
                    returned += 1
        return returned


stock = ShardedAtomicStock(10, shards=2)
store = ReservationStore(stock)

# 10 users reserve immediately (exhausts stock)
ok = [store.reserve(f"r-{i}", user_id=i, ttl_s=0.2) for i in range(12)]
print("reserved?      :", ok)                     # 10 True, then 2 False
print("stock after    :", stock.total)            # 0

# 3 users pay before TTL
for i in (0, 1, 2):
    print(f"pay r-{i}       :", store.pay(f"r-{i}"))

# Wait for TTL to expire the rest
time.sleep(0.3)
returned = store.reap()
print(f"reaper returned: {returned} units")
print(f"stock now      : {stock.total}   # 10 paid=3 → 7 came back")
assert returned == 7 and stock.total == 7
print("✓ expired reservations returned stock")


reserved?      : [True, True, True, True, True, True, True, True, True, True, False, False]
stock after    : 0
pay r-0       : paid
pay r-1       : paid
pay r-2       : paid


reaper returned: 7 units
stock now      : 7   # 10 paid=3 → 7 came back
✓ expired reservations returned stock


## 🎒 Putting it all together (V4)

A production request goes through roughly:

```
client → CDN static page (before start_at)
       → waiting-room token check          (stateless JWT, edge)
       → rate limit (token bucket)         ← TokenBucket class
       → admission queue (bounded)         ← BoundedAdmission class
       → consumer: atomic stock decrement  ← ShardedAtomicStock
       → reservation created with TTL      ← ReservationStore
       → async payment call
       → reaper returns stock on expiry
```

Every piece we built in this notebook shows up on that path.


## 🧪 What we haven't covered (and where to go next)

These would each be their own lab, but they matter:

- **Bot / abuse mitigation**: CAPTCHAs, proof-of-work tokens, device fingerprinting, per-card
  caps. Your best customers and your worst bots look identical on the wire.
- **Geographic partitioning**: some sales shard stock by region (EU/US/APAC) so each region
  has its own Redis and never talks to another.
- **Multi-DC fail-over**: if the primary Redis dies mid-sale, the replica may be seconds behind.
  That's a potential oversell on fail-over. Real answer: accept a small reconciliation window
  and refund/cancel the losing orders post-sale.
- **Observability**: you need per-second counters of reserved, sold_out, 429, 503, queue depth.
  Dashboards + alerting on "queue saturation > 95%" tell you the sale is basically over.
- **Fairness tweaks**: lottery among queued users (random pick) is sometimes fairer than
  pure FIFO — it neutralizes geographic latency advantage.

## 📚 Real-world lessons (headlines)

- **Ticketmaster × Taylor Swift (Nov 2022)**: bots and real users exceeded capacity ~14x.
  The failure mode was the *queue leaking users who had already bought* back into the funnel,
  amplifying load.
- **Alibaba 11/11**: handles ~583k orders/sec by sharding inventory across hundreds of Redis
  instances and doing inventory reconciliation in the background.
- **Xiaomi drops**: classic waiting-room + lottery. "You got picked" is a server decision,
  not a race, which reduces the F5-spam incentive.
- **Supreme**: intentionally small stock + lottery app. The engineering is simple because
  the business model already accepts 99% losers.

The pattern is always: **push the crowd outward, keep stock in exactly one atomic place,
reserve now & pay later, expire TTL to reclaim stock**.
